In [ ]:
import pandas as pd

df = pd.read_csv("all_datasets_aligned.csv")
print(df.head())          # Ver as primeiras linhas
print(df['flag'].value_counts())  # Quantos exemplos tem de cada classe


In [ ]:
import pandas as pd

# Defina o n?mero de amostras desejado para "BENIGN"
n_samples_benign = 1000_000  # ou 1_000_000, como preferir

# Separe a classe majorit?ria
benign_df = df[df['flag'] == 'BENIGN']

# Separe as demais classes (minorit?rias)
minor_classes_df = df[df['flag'] != 'BENIGN']

# Amostragem aleat?ria da classe BENIGN
benign_sampled = benign_df.sample(n=n_samples_benign, random_state=42)

# Combine novamente o DataFrame balanceado
df_balanced = pd.concat([benign_sampled, minor_classes_df], axis=0)

# Embaralhar o DataFrame final
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

# Verificar a nova distribui??o
print(df_balanced['flag'].value_counts())

df = df_balanced
df.to_csv("all_datasets_aligned_balanced.csv", index=False)
del df_balanced, benign_df, minor_classes_df, benign_sampled

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
df = df.dropna()

# Remove a coluna 'ID'
df = df.drop(columns=['id'])




In [ ]:
print(df['flag'].value_counts())


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Codificar r?tulos em n?meros
le = LabelEncoder()
df['flag'] = le.fit_transform(df['flag'])

X = df.drop(columns=['flag'])  # Features
y = df['flag']                 # R?tulos

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=40)

del df  # Liberar mem?ria

In [ ]:
# from sklearn.model_selection import GridSearchCV, StratifiedKFold
# from sklearn.neural_network import MLPClassifier
# from sklearn.metrics import classification_report, accuracy_score

# # Definir o classificador base
# mlp = MLPClassifier(random_state=42, early_stopping=True)

# # Grade de hiperpar?metros
# param_grid = {
#     'hidden_layer_sizes': [(30,), (50,), (30, 30)],
#     'activation': ['relu', 'tanh'],
#     'alpha': [0.0001, 0.01, 0.1],
#     'learning_rate_init': [0.001, 0.01],
#     'solver': ['adam', 'sgd'],
#     'max_iter': [300]
# }

# # Valida??o cruzada estratificada
# cv = StratifiedKFold(n_splits=2, shuffle=True, random_state=42)

# # Executa a busca
# grid_search = GridSearchCV(estimator=mlp, param_grid=param_grid, 
#                            scoring='accuracy', cv=cv, n_jobs=1, verbose=3)

# grid_search.fit(X, y)

# # Resultados
# print("Melhores par?metros encontrados:")
# print(grid_search.best_params_)

# print(f"Melhor acur?cia: {grid_search.best_score_:.4f}")


In [ ]:
# from sklearn.model_selection import cross_val_score, StratifiedKFold
# from sklearn.neural_network import MLPClassifier

# mlp_cv = MLPClassifier(
#     hidden_layer_sizes=(30, 30),
#     alpha=0.0001,
#     activation='relu',
#     max_iter=300,
#     solver='adam',
#     random_state=42,
#     early_stopping=True
# )

# cv10 = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
# scores_10cv = cross_val_score(mlp_cv, X, y, cv=cv10, scoring='accuracy')

# print(f"Acur?cia m?dia (10x CV): {scores_10cv.mean():.4f} ? {scores_10cv.std():.4f}")

In [ ]:
# from sklearn.metrics import classification_report

# # Treinar o modelo com os dados de treino
# mlp_model = MLPClassifier(
#     hidden_layer_sizes=(30, 30),
#     alpha=0.0001,
#     activation='relu',
#     max_iter=300,
#     solver='adam',
#     random_state=42,
#     early_stopping=True
# )
# mlp_model.fit(X_train, y_train)

# # Prever nos dados de teste
# y_pred = mlp_model.predict(X_test)

# # Exibir relat?rio de classifica??o com 4 d?gitos de precis?o
# print(classification_report(y_test, y_pred, target_names=le.classes_.astype(str), digits=5))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import log_loss, accuracy_score, classification_report

# Par?metros
n_epochs = 25
classes = np.unique(y_train)

# Criar o modelo com warm_start=True para treinar por ?poca
mlp_model = MLPClassifier(
    hidden_layer_sizes=(30, 30),
    alpha=0.0001,
    activation='relu',
    solver='adam',
    max_iter=1,
    warm_start=True,
    random_state=42
)

# Listas para armazenar m?tricas por ?poca
train_loss_list = []
test_loss_list = []
train_acc_list = []
test_acc_list = []

# Treinamento manual por ?poca
for epoch in range(n_epochs):
    mlp_model.fit(X_train, y_train)

    # Probabilidades para calcular log loss
    y_train_proba = mlp_model.predict_proba(X_train)
    y_test_proba = mlp_model.predict_proba(X_test)

    # Loss
    train_loss = log_loss(y_train, y_train_proba)
    test_loss = log_loss(y_test, y_test_proba)

    # Acur?cia
    train_acc = accuracy_score(y_train, mlp_model.predict(X_train))
    test_acc = accuracy_score(y_test, mlp_model.predict(X_test))

    # Armazenar
    train_loss_list.append(train_loss)
    test_loss_list.append(test_loss)
    train_acc_list.append(train_acc)
    test_acc_list.append(test_acc)

# Ap?s treino, imprimir relat?rio final
y_pred = mlp_model.predict(X_test)
print(classification_report(y_test, y_pred, target_names=le.classes_.astype(str), digits=5))


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(train_loss_list, label='Loss de Treino', color='blue')
plt.plot(test_loss_list, label='Loss de Teste', color='red')
plt.xlabel('?poca')
plt.ylabel('Loss')
plt.title('Loss de Treino e Teste - MLPClassifier')
plt.legend()
plt.grid()
plt.show()


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(train_acc_list, label='Acur?cia de Treino', color='blue')
plt.plot(test_acc_list, label='Acur?cia de Teste', color='orange')
plt.xlabel('?poca')
plt.ylabel('Acur?cia')
plt.title('Acur?cia de Treino e Teste - MLPClassifier')
plt.legend()
plt.grid()
plt.show()


In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report, roc_curve, auc
from sklearn.preprocessing import label_binarize
import seaborn as sns
# --- Matriz de Confus?o ---
conf_matrix = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues", xticklabels=le.classes_, yticklabels=le.classes_)
plt.xlabel("Previsto")
plt.ylabel("Real")
plt.title("Matriz de Confus?o")
plt.tight_layout()
plt.show()

#   y_train_proba = mlp_model.predict_proba(X_train)
#     y_test_proba = mlp_model.predict_proba(X_test)
# --- Tabela de M?tricas ---
report = classification_report(y_test, y_pred, target_names=le.classes_.astype(str), output_dict=True)
report_df = pd.DataFrame(report).transpose()
report_df = report_df.round(4)
report_df

# --- Curva ROC para Multiclasse ---
# Binariza as sa?das
y_test_bin = label_binarize(y_test, classes=np.arange(len(le.classes_)))
n_classes = y_test_bin.shape[1]

# Calcular curva ROC e AUC para cada classe
fpr = dict()
tpr = dict()
roc_auc = dict()
for i in range(n_classes):
    fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], y_test_proba[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

# Plotar todas as curvas ROC
plt.figure(figsize=(8, 6))
for i in range(n_classes):
    plt.plot(fpr[i], tpr[i], label=f"Classe {le.classes_[i]} (AUC = {roc_auc[i]:.4})")
plt.plot([0, 1], [0, 1], "k--", lw=1)
plt.xlabel("Taxa de Falsos Positivos")
plt.ylabel("Taxa de Verdadeiros Positivos")
plt.title("Curva ROC - Multiclasse")
plt.legend(loc="lower right")
plt.grid()
plt.tight_layout()
plt.show()